# Week 2, Notebook 1: Your First PyTorch Network
## From Scratch to Framework — Feel the Difference

**What you'll build:** Rebuild your Week 1 network in PyTorch, then beat it.

**Curriculum points:**
- ⑤ Backprop → PyTorch autograd does it for you
- ⑦ Capacity ≠ performance → architecture matters
- ② ReLU → compare activations in a real framework

**Time estimate:** 45–60 minutes

---
### Why PyTorch?
You built everything from scratch in Week 1. Now you understand what the framework does.
PyTorch's `autograd` computes backprop automatically — the same chain rule, just automated.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Part 1: Rebuild Week 1's Network in PyTorch

In [ ]:
# ============================================================
# Generate the same moons dataset
# ============================================================
def make_moons_torch(n=500, noise=0.15):
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.c_[np.cos(t), np.sin(t)] + np.random.randn(n // 2, 2) * noise
    x2 = np.c_[np.cos(t) + 0.5, -np.sin(t) + 0.5] + np.random.randn(n // 2, 2) * noise
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n // 2), np.ones(n // 2)])
    idx = np.random.permutation(n)
    X, y = X[idx], y[idx]
    
    # Normalize
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)
    
    # Convert to PyTorch tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).unsqueeze(1)
    return X_tensor, y_tensor

X_all, y_all = make_moons_torch(600)

# Train/val split
n_train = 480
X_train, y_train = X_all[:n_train], y_all[:n_train]
X_val, y_val = X_all[n_train:], y_all[n_train:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

In [ ]:
# ============================================================
# PyTorch network — compare with your pure Python version!
# ============================================================

class SimpleNet(nn.Module):
    """Same architecture as Week 1, but in PyTorch."""
    
    def __init__(self, hidden_size=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

# Compare: your pure Python version was ~50 lines of Layer class
# PyTorch version: 6 lines. Same result.
model = SimpleNet(16)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# ============================================================
# Training loop — PyTorch handles backprop via autograd!
# ============================================================
def train_model(model, X_train, y_train, X_val, y_val, 
                lr=0.01, epochs=1000, optimizer_class=optim.SGD):
    """Generic training loop."""
    optimizer = optimizer_class(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        pred_train = model(X_train)
        loss_train = criterion(pred_train, y_train)
        
        optimizer.zero_grad()    # Clear old gradients
        loss_train.backward()    # Compute gradients (autograd = chain rule!)
        optimizer.step()         # Update weights
        
        # --- Validation (no gradients needed) ---
        model.eval()
        with torch.no_grad():
            pred_val = model(X_val)
            loss_val = criterion(pred_val, y_val)
            
            acc_train = ((pred_train > 0.5).float() == y_train).float().mean()
            acc_val = ((pred_val > 0.5).float() == y_val).float().mean()
        
        history['train_loss'].append(loss_train.item())
        history['val_loss'].append(loss_val.item())
        history['train_acc'].append(acc_train.item())
        history['val_acc'].append(acc_val.item())
        
        if epoch % 200 == 0:
            print(f"  Epoch {epoch:4d} | Train Loss: {loss_train.item():.4f} | "
                  f"Val Acc: {acc_val.item():.3f}")
    
    return history

# Train!
model = SimpleNet(16)
history = train_model(model, X_train, y_train, X_val, y_val, lr=0.05, epochs=1500)

print(f"\nFinal: Train Acc={history['train_acc'][-1]:.3f}, Val Acc={history['val_acc'][-1]:.3f}")

In [ ]:
# Visualize results
def plot_results(model, X_val, y_val, history, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Loss curves
    axes[0].plot(history['train_loss'], label='Train', alpha=0.8)
    axes[0].plot(history['val_loss'], label='Val', alpha=0.8)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].set_title('Loss')
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.2)
    
    # Accuracy curves
    axes[1].plot(history['train_acc'], label='Train', alpha=0.8)
    axes[1].plot(history['val_acc'], label='Val', alpha=0.8)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].set_title('Accuracy')
    axes[1].set_ylim(0.4, 1.05)
    axes[1].grid(True, alpha=0.2)
    
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
    grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
    model.eval()
    with torch.no_grad():
        zz = model(grid).numpy().reshape(xx.shape)
    
    axes[2].contourf(xx, yy, zz, levels=20, cmap='RdYlBu_r', alpha=0.7)
    axes[2].contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2)
    X_np = X_val.numpy()
    y_np = y_val.numpy().flatten()
    axes[2].scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='steelblue', s=25, alpha=0.6)
    axes[2].scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='coral', s=25, alpha=0.6)
    axes[2].set_title('Decision Boundary')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

fig = plot_results(model, X_val, y_val, history, "PyTorch SimpleNet — Moons")
plt.savefig('w2_01_pytorch_basic.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 2: SGD vs Adam — Optimizer Comparison

Different optimizers navigate the loss landscape differently.  
Adam adapts learning rates per-parameter — usually converges faster.

In [ ]:
# ============================================================
# EXPERIMENT: SGD vs Adam vs SGD+Momentum
# ============================================================
optimizers = {
    'SGD (lr=0.05)': (optim.SGD, {'lr': 0.05}),
    'SGD+Momentum': (optim.SGD, {'lr': 0.05, 'momentum': 0.9}),
    'Adam (lr=0.01)': (optim.Adam, {'lr': 0.01}),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['steelblue', 'coral', 'forestgreen']

for (name, (opt_cls, opt_kwargs)), color in zip(optimizers.items(), colors):
    torch.manual_seed(42)
    model = SimpleNet(16)
    optimizer = opt_cls(model.parameters(), **opt_kwargs)
    criterion = nn.MSELoss()
    
    losses = []
    accs = []
    for epoch in range(1500):
        model.train()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        model.eval()
        with torch.no_grad():
            acc = ((model(X_val) > 0.5).float() == y_val).float().mean().item()
            accs.append(acc)
    
    axes[0].plot(losses, label=name, color=color, alpha=0.8)
    axes[1].plot(accs, label=name, color=color, alpha=0.8)
    
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-3, 3, 150), np.linspace(-3, 3, 150))
    grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
    with torch.no_grad():
        zz = model(grid).numpy().reshape(xx.shape)
    axes[2].contour(xx, yy, zz, levels=[0.5], colors=color, linewidths=2)

axes[0].set_title('Training Loss')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.2)

axes[1].set_title('Validation Accuracy')
axes[1].set_ylim(0.5, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.2)

# Add data points to boundary plot
X_np = X_val.numpy()
y_np = y_val.numpy().flatten()
axes[2].scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='steelblue', s=15, alpha=0.3)
axes[2].scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='coral', s=15, alpha=0.3)
axes[2].set_title('Decision Boundaries (all 3)')
axes[2].legend(['SGD', 'SGD+Mom', 'Adam'])

plt.suptitle('OPTIMIZER COMPARISON', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w2_01_optimizers.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n⚡ KEY INSIGHT: Adam usually converges faster, but all reach similar final accuracy.")
print("   Momentum helps SGD escape shallow local minima.")

## Part 3: Autograd Deep Dive — What PyTorch Does For You

Let's peek under the hood to confirm PyTorch's autograd matches our manual backprop.

In [ ]:
# ============================================================
# Peek at autograd — see the gradients PyTorch computes
# ============================================================
torch.manual_seed(42)
model = SimpleNet(4)  # small for visibility

# Forward pass
x_sample = X_train[:1]  # single sample
y_sample = y_train[:1]

pred = model(x_sample)
loss = nn.MSELoss()(pred, y_sample)

print("Forward pass:")
print(f"  Input: {x_sample.data}")
print(f"  Prediction: {pred.item():.4f}")
print(f"  Target: {y_sample.item():.1f}")
print(f"  Loss: {loss.item():.6f}")

# Backward pass
loss.backward()

print("\nGradients (computed by autograd = chain rule):")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name:20s} shape={str(list(param.shape)):15s} "
              f"grad_mean={param.grad.mean().item():+.6f}")

print("\n✓ These are EXACTLY the same gradients you computed manually in Week 1!")
print("  autograd = automatic chain rule application = backpropagation")

## ✅ Self-Check

- [ ] You can write a PyTorch training loop from memory
- [ ] You understand that `loss.backward()` = your manual chain rule from Week 1
- [ ] You can compare optimizers and predict Adam will converge faster
- [ ] You can explain what `model.eval()` and `torch.no_grad()` do and why

## ➡️ Next: `W2_02_PyTorch_Going_Deeper.ipynb` — Init, normalization, loss landscapes

## Visualizing the PyTorch Computational Graph

A diagram of the simple sequential network built with `nn.Sequential`.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
layers = {'Input': ['in_1', 'in_2'], 'Linear1': ['l1_1', 'l1_2', 'l1_3'], 'ReLU1': ['r1', 'r2', 'r3'], 'Linear2': ['out']}
pos = {}
for i, (layer, nodes) in enumerate(layers.items()):
    for j, node in enumerate(nodes):
        pos[node] = (i * 2, len(nodes) / 2.0 - j)
        G.add_node(node)

for u in layers['Input']:
    for v in layers['Linear1']: G.add_edge(u, v)
for u, v in zip(layers['Linear1'], layers['ReLU1']):
    G.add_edge(u, v)
for u in layers['ReLU1']:
    for v in layers['Linear2']: G.add_edge(u, v)

plt.figure(figsize=(10, 4))
nx.draw(G, pos, with_labels=True, node_color='plum', node_size=1200, arrowsize=15)
plt.title("PyTorch Sequential Network Graph")

plt.savefig('w2_01_network_arch.png', dpi=100, bbox_inches='tight')
plt.show()
